<a href="https://colab.research.google.com/github/Chandu1722/ML/blob/main/Kaggle/LoanPayBack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [125]:
import pandas as pd

In [126]:
train=pd.read_csv('train.csv')
test=pd.read_csv('test.csv')

In [127]:
train.head()

,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade,loan_paid_back
0,0,29367.99,0.084,736,2528.42,13.67,Female,Single,High School,Self-employed,Other,C3,1.0
1,1,22108.02,0.166,636,4593.10,12.92,Male,Married,Master's,Employed,Debt consolidation,D3,0.0
2,2,49566.20,0.097,694,17005.15,9.76,Male,Single,High School,Employed,Debt consolidation,C5,1.0
3,3,46858.25,0.065,533,4682.48,16.10,Female,Single,High School,Employed,Debt consolidation,F1,1.0
4,4,25496.70,0.053,665,12184.43,10.21,Male,Married,High School,Employed,Other,D1,1.0


In [128]:
# --- Add this code in a new cell after you load 'train.csv' ---

# 1. Create loan_to_income_ratio
# (Adding 1 to income to prevent any chance of dividing by zero)
train['loan_to_income_ratio'] = train['loan_amount'] / (train['annual_income'] + 1)
test['loan_to_income_ratio'] = test['loan_amount'] / (test['annual_income'] + 1)

# 2. Create total_debt
# This calculates the actual debt amount from the ratio
train['total_debt'] = train['debt_to_income_ratio'] * train['annual_income']
test['total_debt'] = test['debt_to_income_ratio'] * test['annual_income']

# 3. Bin credit scores into categories
# This can help models capture non-linear relationships
score_bins = [0, 579, 669, 739, 799, 850]
score_labels = ['Poor', 'Fair', 'Good', 'Very_Good', 'Excellent']
train['credit_score_category'] = pd.cut(train['credit_score'], bins=score_bins, labels=score_labels)
test['credit_score_category'] = pd.cut(test['credit_score'], bins=score_bins, labels=score_labels)

In [129]:
train.columns

Index(['id', 'annual_income', 'debt_to_income_ratio', 'credit_score',
       'loan_amount', 'interest_rate', 'gender', 'marital_status',
       'education_level', 'employment_status', 'loan_purpose',
       'grade_subgrade', 'loan_paid_back', 'loan_to_income_ratio',
       'total_debt', 'credit_score_category'],
      dtype='object')

In [130]:
train['grade_subgrade'].unique()

array(['C3', 'D3', 'C5', 'F1', 'D1', 'D5', 'C2', 'C1', 'F5', 'D4', 'C4',
       'D2', 'E5', 'B1', 'B2', 'F4', 'A4', 'E1', 'F2', 'B4', 'E4', 'B3',
       'E3', 'B5', 'E2', 'F3', 'A5', 'A3', 'A1', 'A2'], dtype=object)

In [105]:
nominal=['marital_status','gender','employment_status','loan_purpose']

In [132]:
ordinal_cols=['education_level','grade_subgrade','credit_score_category']
seq_order=[['High School', "Bachelor's", "Master's",'PhD', 'Other'],
           ['A1', 'A2', 'A3', 'A4', 'A5',
            'B1', 'B2', 'B3', 'B4', 'B5',
            'C1', 'C2', 'C3', 'C4', 'C5',
            'D1', 'D2', 'D3', 'D4', 'D5',
            'E1', 'E2', 'E3', 'E4', 'E5',
            'F1', 'F2', 'F3', 'F4', 'F5'],
           ['Poor', 'Fair', 'Good', 'Very_Good', 'Excellent']]

In [133]:
numerical_cols=['annual_income','credit_score','loan_amount','interest_rate','debt_to_income_ratio','total_debt','loan_to_income_ratio']

In [134]:
train=pd.get_dummies(train,columns=nominal)
test=pd.get_dummies(test,columns=nominal)

In [135]:
from sklearn.preprocessing import OrdinalEncoder
encoder=OrdinalEncoder(categories=seq_order)
train[ordinal_cols]=encoder.fit_transform(train[ordinal_cols])
test[ordinal_cols]=encoder.transform(test[ordinal_cols])

In [136]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler(copy=False)
train[numerical_cols]=scaler.fit_transform(train[numerical_cols])
test[numerical_cols]=scaler.transform(test[numerical_cols])

In [137]:
train.head()

,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,education_level,grade_subgrade,loan_paid_back,loan_to_income_ratio,...,employment_status_Student,employment_status_Unemployed,loan_purpose_Business,loan_purpose_Car,loan_purpose_Debt consolidation,loan_purpose_Education,loan_purpose_Home,loan_purpose_Medical,loan_purpose_Other,loan_purpose_Vacation
0,0,-0.705461,-0.535135,0.993849,-1.803484,0.653899,0.0,12.0,1.0,-0.932934,...,False,False,False,False,False,False,False,False,True,False
1,1,-0.977248,0.660668,-0.810394,-1.505401,0.280571,2.0,17.0,0.0,-0.600551,...,False,False,False,False,True,False,False,False,False,False
2,2,0.050689,-0.345556,0.236067,0.286558,-1.292385,0.0,14.0,1.0,-0.230824,...,False,False,False,False,True,False,False,False,False,False
3,3,-0.050687,-0.812211,-2.668764,-1.492497,1.863482,0.0,25.0,1.0,-0.895135,...,False,False,False,False,True,False,False,False,False,False
4,4,-0.850388,-0.987206,-0.287163,-0.409421,-1.068388,0.0,15.0,1.0,0.137445,...,False,False,False,False,False,False,False,False,True,False


In [138]:
X=train.drop(columns=['id','loan_paid_back'])
y=train['loan_paid_back']
y_test_final=test.drop(columns=['id'])


In [139]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [144]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train,y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [145]:
from sklearn.metrics import roc_auc_score
y_pred_lo=model.predict_proba(X_test)[:, 1]
auc=roc_auc_score(y_test,y_pred_lo)
print(auc)

0.9112580067072493


In [146]:
from xgboost import XGBClassifier
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model_xgb = XGBClassifier(scale_pos_weight=scale_pos_weight)
model_xgb.fit(X_train,y_train)
y_pred_xgb=model_xgb.predict_proba(X_test)[:, 1]

auc=roc_auc_score(y_test,y_pred_xgb)
print(auc)

0.9210269533658304


In [147]:
model_xgb.fit(X,y)
y_pred_xgb=model_xgb.predict_proba(y_test_final)[:, 1]
output=pd.DataFrame({'id':test['id'],'loan_paid_back':y_pred_xgb})
output.to_csv('submissionloan3.csv', index=False)

In [143]:
from lightgbm import LGBMClassifier
model_lgbm=LGBMClassifier(is_unbalance=True)
model_lgbm.fit(X_train,y_train)
y_pred_lgbm=model_lgbm.predict_proba(X_test)[:, 1]
auc=roc_auc_score(y_test,y_pred_lgbm)
print(auc)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 379692, number of negative: 95503
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.047849 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1864
[LightGBM] [Info] Number of data points in the train set: 475195, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.799024 -> initscore=1.380203
[LightGBM] [Info] Start training from score 1.380203
0.9208894852319119


In [123]:
from lightgbm import LGBMClassifier
model_lgbm=LGBMClassifier(is_unbalance=True)
model_lgbm.fit(X,y)
y_pred_lgbm=model_lgbm.predict_proba(y_test_final)[:, 1]

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 474494, number of negative: 119500
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.141191 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1864
[LightGBM] [Info] Number of data points in the train set: 593994, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.798820 -> initscore=1.378933
[LightGBM] [Info] Start training from score 1.378933


In [124]:
output=pd.DataFrame({'id':test['id'],'loan_paid_back':y_pred_lgbm})
output.to_csv('submissionLoan2.csv', index=False)